In [ ]:
import datetime
import numpy as np
np.set_printoptions(precision=4)
from numpy.typing import NDArray
from typing import Optional
import pandas as pd

sym = 'NVDA' # NVDA 2025-02-07
dayoffset = -4
date = f'{datetime.datetime.now()+datetime.timedelta(days=dayoffset):%Y%m%d}'
filename = f'{sym}_5s_{date}.csv'
load_from_csv = True
bars = None
if load_from_csv:
    bars_df = pd.read_csv(rf'.\data\{filename}', parse_dates=['date'])
if 'open_' in bars_df.columns:
    bars_df.rename(columns={'open_': 'open'}, inplace=True)
bars_df['log_avg'] = np.log(bars_df['average'])
bars_df[['open', 'high', 'low', 'close', 'average', 'date']].head()

In [ ]:
from collections import deque

class OnlineAnomalyDetector:
    def __init__(self, window_size=100, threshold=7):
        self.window_size = window_size
        self.threshold = threshold
        self.data_window = deque(maxlen=window_size)
        self.anomaly_count = deque(maxlen=window_size) # parallel to data_window
        # self.mean = np.zeros(4) # assume zero mean
        self.var = np.zeros(4) # sum of squares
        self.n = 0 # number of samples included in sum of squares

    def update_stats(self, diff: NDArray):
        self.n += 1
        if self.n == 1:
            # self.mean = diff
            self.var = np.zeros(4)
        else:
            # old_mean = self.mean.copy()
            # self.mean += (diff - old_mean) / self.n
            # self.var += (diff - old_mean) * (diff - self.mean)
            self.var += diff * diff

    def undo_update_stats(self, diff: NDArray):
        """
        Undo the update of stats for the previous data point because it was an anomaly
        """
        self.n -= 1
        if self.n == 0:
            self.var = np.zeros(4)
        else:
            self.var -= diff * diff

    def detect_anomaly(self, current_data, /, update_stats=True):
        if len(self.data_window) < 2:
            self.data_window.append(current_data)
            return False, []

        self.prev_data = prev_data = self.data_window[-1]
        diff = np.array(current_data) - np.array(prev_data)

        self.update_stats(diff)

        if self.n > 1:
            masked_var = np.ma.masked_values(self.var, 0.) # ignore zero variance
            std_dev = np.sqrt(masked_var / (self.n - 1))
            # print(f"std_dev: {std_dev.data}, abs diff: {np.abs(diff)}")
            # z_scores = np.abs(diff - self.mean) / std_dev
            z_scores = np.abs(diff) / std_dev

            anomalies = np.where(z_scores > self.threshold)[0]
            is_anomaly = len(anomalies) > 0
            if is_anomaly:
                self.undo_update_stats(diff)
                # print(f"Anomaly detected: {current_data}")
                # print(f"Mean: {self.mean}, Std Dev: {std_dev}")
                print(f"Diff: {diff}, std_dev: {std_dev.data}, z_scores: {z_scores.data}")
                # print(f"Anomalies: {anomalies}")

            self.data_window.append(current_data)
            return is_anomaly, anomalies
        else:
            self.data_window.append(current_data)
            return False, []
    
    def get_prev_data(self):
        return self.prev_data


In [ ]:
# Example usage
detector = OnlineAnomalyDetector(window_size=100, threshold=15)

def process_data_point(data_point, /, index: Optional[pd.Timestamp]=None):
    is_anomaly, anomalous_features = detector.detect_anomaly(data_point, update_stats=True)
    if is_anomaly and (index is None or index.time() >= datetime.time(9, 30)):
        feature_names = ['Open', 'High', 'Low', 'Close']
        anomalous_features = [feature_names[i] for i in anomalous_features]
        print(f"Anomaly detected in: {', '.join(anomalous_features)}")
        print(f"Data point: {data_point}" + (f" (index: {index})" if index is not None else ""))
        print(f"Previous data point: {detector.get_prev_data()}")
    return is_anomaly



In [ ]:
# Simulating data stream
import random
import time

prev_data_point = None
for row in bars_df[['open', 'high', 'low', 'close', 'date']].set_index('date').itertuples(index=True, name=None):
    # print(f"Processed data point: {row}")
    index, *data_point = row
    process_data_point(data_point, index)
    


In [ ]:
# # Simulating data stream
# import random
# import time

# while True:
#     # Generate sample data (replace this with your actual data stream)
#     data_point = tuple(random.uniform(0, 100) for _ in range(4))
    
#     # Occasionally introduce an anomaly
#     if random.random() < 0.05:
#         index = random.randint(0, 3)
#         data_point = list(data_point)
#         data_point[index] *= random.uniform(1.5, 2.0)  # Increase one value significantly
#         data_point = tuple(data_point)

#     process_data_point(data_point)
#     print(f"Processed data point: {data_point}")
#     time.sleep(.1)  # Wait for 5 seconds before the next data point
